# 01 — Base municipale

Source unique : `exctraction of data/output/CSV_final.csv`.

Une ligne = une municipalité, retenue si `MUNICIPIO/TIOC` est non vide (les lignes vides sont
des sous-totaux département / province / total "Amazonía Norte").

- `pop_2001/2012/2024` <- bloc "NÚMERO DE PERSONAS POR FUENTE DE ELECTRICIDAD", sous-colonne "Total"
- `hh_2001/2012/2024` <- bloc "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD", sous-colonne
  "Electricidad total" (2001) ou "Total" (2012, 2024)
- `hhsize_YYYY = pop_YYYY / hh_YYYY`
- `r_pop_0112, r_hh_0112 = 100 * ln(x_2012/x_2001) / 11` (taux annuel %/an), idem `r_..._1224` sur 2012-2024

Aucun index en dur : les colonnes sont repérées par leur en-tête exact.

In [ ]:
import math
import os

import pandas as pd

BASE = os.path.dirname(os.getcwd())
CSV_PATH = os.path.join(BASE, "exctraction of data", "output", "CSV_final.csv")

POP_BLOCK = "NÚMERO DE PERSONAS POR FUENTE DE ELECTRICIDAD"
HH_BLOCK = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD"

POP_SUBCOL_BY_YEAR = {2001: "Total", 2012: "Total", 2024: "Total"}
HH_SUBCOL_BY_YEAR = {2001: "Electricidad total", 2012: "Total", 2024: "Total"}

EXPECTED_TOTAL_ROW_LABEL = "Amazonía Norte"
EXPECTED_N_MUNICIPIOS = 21
EXPECTED_POP_TOTALS = {2001: 194207, 2012: 267766, 2024: 317517}
EXPECTED_HH_TOTALS = {2001: 37569, 2012: 62360, 2024: 84209}


def find_column(raw_columns, block, year, subcol):
    target = f"{block} | {year} | {subcol}"
    matches = [c for c in raw_columns if c == target]
    assert len(matches) == 1, f"Colonne introuvable ou ambiguë : {target!r} ({len(matches)} correspondance(s))"
    return matches[0]

## 1. Lecture et filtre des municipalités

In [ ]:
raw = pd.read_csv(CSV_PATH, encoding="utf-8")
raw_columns = list(raw.columns)

total_row = raw[raw["MUNICIPIO/TIOC"].isna() & (raw["DEPARTAMENTO"] == EXPECTED_TOTAL_ROW_LABEL)]
municipios = raw[raw["MUNICIPIO/TIOC"].notna() & (raw["MUNICIPIO/TIOC"].astype(str).str.strip() != "")].copy()

assert len(municipios) == EXPECTED_N_MUNICIPIOS, (
    f"Nombre de municipalités retenues = {len(municipios)}, attendu {EXPECTED_N_MUNICIPIOS}. "
    "Vérifier le filtre MUNICIPIO/TIOC non vide (colonnes ou lecture décalées)."
)

out = municipios[["DEPARTAMENTO", "PROVINCIA", "MUNICIPIO/TIOC"]].rename(
    columns={"DEPARTAMENTO": "departamento", "PROVINCIA": "provincia", "MUNICIPIO/TIOC": "municipio"}
).reset_index(drop=True)
municipios = municipios.reset_index(drop=True)
out

,departamento,provincia,municipio
0,La Paz,Abel Iturralde,Ixiamas
1,Beni,Vaca Diez,Riberalta
2,Beni,Vaca Diez,Guayaramerín
3,Beni,General José Ballivián,Reyes
4,Beni,General José Ballivián,Santa Rosa
5,Beni,Yacuma,Exaltación
6,Pando,Nicolás Suárez,Cobija
7,Pando,Nicolás Suárez,Porvenir
8,Pando,Nicolás Suárez,Bolpebra
9,Pando,Nicolás Suárez,Bella Flor


## 2. Colonnes pop / hh par année

In [ ]:
for year, subcol in POP_SUBCOL_BY_YEAR.items():
    col = find_column(raw_columns, POP_BLOCK, year, subcol)
    out[f"pop_{year}"] = municipios[col].astype(float)

for year, subcol in HH_SUBCOL_BY_YEAR.items():
    col = find_column(raw_columns, HH_BLOCK, year, subcol)
    out[f"hh_{year}"] = municipios[col].astype(float)

out

,departamento,provincia,municipio,pop_2001,pop_2012,pop_2024,hh_2001,hh_2012,hh_2024
0,La Paz,Abel Iturralde,Ixiamas,5207.0,8338.0,11330.0,1197.0,2270.0,3306.0
1,Beni,Vaca Diez,Riberalta,73981.0,86903.0,107816.0,13558.0,19588.0,27442.0
2,Beni,Vaca Diez,Guayaramerín,39173.0,40186.0,40130.0,7815.0,9261.0,10891.0
3,Beni,General José Ballivián,Reyes,11016.0,12912.0,11284.0,1815.0,2962.0,3417.0
4,Beni,General José Ballivián,Santa Rosa,8915.0,9297.0,10953.0,1567.0,1976.0,2755.0
5,Beni,Yacuma,Exaltación,6504.0,5996.0,7810.0,1090.0,938.0,1455.0
6,Pando,Nicolás Suárez,Cobija,20873.0,43616.0,51908.0,4923.0,12304.0,15564.0
7,Pando,Nicolás Suárez,Porvenir,3554.0,7653.0,9096.0,831.0,1655.0,2230.0
8,Pando,Nicolás Suárez,Bolpebra,1137.0,2058.0,2338.0,292.0,545.0,802.0
9,Pando,Nicolás Suárez,Bella Flor,2140.0,3550.0,3421.0,497.0,944.0,1235.0


## 3. Colonnes calculées — taille des ménages et taux de croissance

In [ ]:
for year in (2001, 2012, 2024):
    out[f"hhsize_{year}"] = out[f"pop_{year}"] / out[f"hh_{year}"]

out["r_pop_0112"] = 100 * (out["pop_2012"] / out["pop_2001"]).apply(math.log) / 11
out["r_pop_1224"] = 100 * (out["pop_2024"] / out["pop_2012"]).apply(math.log) / 12
out["r_hh_0112"] = 100 * (out["hh_2012"] / out["hh_2001"]).apply(math.log) / 11
out["r_hh_1224"] = 100 * (out["hh_2024"] / out["hh_2012"]).apply(math.log) / 12
out

,departamento,provincia,municipio,pop_2001,pop_2012,pop_2024,hh_2001,hh_2012,hh_2024,hhsize_2001,hhsize_2012,hhsize_2024,r_pop_0112,r_pop_1224,r_hh_0112,r_hh_1224
0,La Paz,Abel Iturralde,Ixiamas,5207.0,8338.0,11330.0,1197.0,2270.0,3306.0,4.350042,3.673128,3.427102,4.280177,2.555256,5.817831,3.132993
1,Beni,Vaca Diez,Riberalta,73981.0,86903.0,107816.0,13558.0,19588.0,27442.0,5.456631,4.436543,3.928868,1.463493,1.796946,3.344912,2.809646
2,Beni,Vaca Diez,Guayaramerín,39173.0,40186.0,40130.0,7815.0,9261.0,10891.0,5.012540,4.339272,3.684694,0.232099,-0.011621,1.543337,1.351039
3,Beni,General José Ballivián,Reyes,11016.0,12912.0,11284.0,1815.0,2962.0,3417.0,6.069421,4.359217,3.302312,1.443712,-1.123094,4.452539,1.190819
4,Beni,General José Ballivián,Santa Rosa,8915.0,9297.0,10953.0,1567.0,1976.0,2755.0,5.689215,4.704960,3.975681,0.381423,1.366014,2.108288,2.769524
5,Beni,Yacuma,Exaltación,6504.0,5996.0,7810.0,1090.0,938.0,1455.0,5.966972,6.392324,5.367698,-0.739316,2.202603,-1.365300,3.658427
6,Pando,Nicolás Suárez,Cobija,20873.0,43616.0,51908.0,4923.0,12304.0,15564.0,4.239894,3.544863,3.335132,6.699705,1.450407,8.327330,1.958635
7,Pando,Nicolás Suárez,Porvenir,3554.0,7653.0,9096.0,831.0,1655.0,2230.0,4.276775,4.624169,4.078924,6.972945,1.439475,6.262968,2.485005
8,Pando,Nicolás Suárez,Bolpebra,1137.0,2058.0,2338.0,292.0,545.0,802.0,3.893836,3.776147,2.915212,5.394013,1.063010,5.673018,3.219357
9,Pando,Nicolás Suárez,Bella Flor,2140.0,3550.0,3421.0,497.0,944.0,1235.0,4.305835,3.760593,2.770040,4.601289,-0.308456,5.832147,2.239167


## 4. Garde-fou

La somme des 21 municipalités doit égaler strictement la ligne "Amazonía Norte" pour pop et hh,
à chaque année. Tout écart signale une colonne mal repérée ou un filtre de lignes incorrect.

In [ ]:
for year in (2001, 2012, 2024):
    pop_sum = out[f"pop_{year}"].sum()
    pop_col = find_column(raw_columns, POP_BLOCK, year, POP_SUBCOL_BY_YEAR[year])
    pop_expected = float(total_row[pop_col].iloc[0])
    assert pop_sum == pop_expected == EXPECTED_POP_TOTALS[year], (
        f"pop_{year}: somme des 21 municipalités = {pop_sum}, "
        f"ligne 'Amazonía Norte' = {pop_expected}, attendu {EXPECTED_POP_TOTALS[year]}"
    )

    hh_sum = out[f"hh_{year}"].sum()
    hh_col = find_column(raw_columns, HH_BLOCK, year, HH_SUBCOL_BY_YEAR[year])
    hh_expected = float(total_row[hh_col].iloc[0])
    assert hh_sum == hh_expected == EXPECTED_HH_TOTALS[year], (
        f"hh_{year}: somme des 21 municipalités = {hh_sum}, "
        f"ligne 'Amazonía Norte' = {hh_expected}, attendu {EXPECTED_HH_TOTALS[year]}"
    )

print("Assertions OK : 21 municipalités retenues, sommes pop/hh = ligne 'Amazonía Norte' pour 2001/2012/2024.")

Assertions OK : 21 municipalités retenues, sommes pop/hh = ligne 'Amazonía Norte' pour 2001/2012/2024.
